In [1]:
pip install rasterio numpy matplotlib pandas seaborn

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install rasterio pandas numpy fiona shapely

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import glob
import re
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
import fiona

# =====================================================================
# PATH CONFIGURATION
# =====================================================================
input_dir = r"E:\2_Contribution\Final\Preprocess\LST"
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML\LST"
os.makedirs(output_dir, exist_ok=True)

# OPTIONAL BUT RECOMMENDED: Path to your Ningxia Shapefile (.shp)
# If you don't have a shapefile, set this to None, and the script will 
# fall back to using the native TIF extent and internal NoData masks.
ningxia_shp_path = None  # Example: r"E:\2_Contribution\Boundary\Ningxia.shp"

# =====================================================================
# STEP 1: TEMPORAL FILTERING (Growing Season 2001-2020)
# =====================================================================
print("Step 1: Scanning directory and applying temporal filtering...")
all_tif_files = glob.glob(os.path.join(input_dir, "*.tif"))
pattern = re.compile(r"LST_(\d{4})_(\d{2})\.tif")

filtered_files = []
for filepath in all_tif_files:
    match = pattern.search(os.path.basename(filepath))
    if match:
        year = int(match.group(1))
        month = int(match.group(2))
        # Growing season (Months 4 to 10) between 2001 and 2020
        if 2001 <= year <= 2020 and 4 <= month <= 10:
            filtered_files.append((filepath, year, month))

print(f"-> Found {len(filtered_files)} files matching the growing season criteria.")

if not filtered_files:
    raise ValueError("No matching TIF files found. Check your directory or file names.")

# =====================================================================
# STEP 2 & 3: SPATIAL ALIGNMENT, BOUNDARY MASKING & EXTRACTION
# =====================================================================
print("\nStep 2 & 3: Aligning spatial grids, masking to Ningxia, and extracting to table...")

# Load vector geometry if shapefile is provided
shapes = None
if ningxia_shp_path and os.path.exists(ningxia_shp_path):
    print(f"-> Loading Ningxia boundary shapefile: {ningxia_shp_path}")
    with fiona.open(ningxia_shp_path, "r") as shapefile:
        shapes = [feature["geometry"] for feature in shapefile]
else:
    print("-> No shapefile provided (or path invalid). Extracting using TIF internal masks.")

all_extracted_records = []

for idx, (filepath, year, month) in enumerate(filtered_files):
    with rasterio.open(filepath) as src:
        # Check spatial resolution sanity check (expecting ~1km / 0.00833 deg)
        res_x, res_y = src.res
        crs = src.crs
        nodata_val = src.nodata
        
        # Apply Shapefile Masking if available
        if shapes is not None:
            try:
                # crop=True ensures the matrix wraps tightly around the Ningxia boundary
                img_data, transform = mask(src, shapes, crop=True, nodata=nodata_val if nodata_val is not None else -9999)
                band_data = img_data[0] # Get first band
            except ValueError:
                print(f"Warning: File {os.path.basename(filepath)} does not overlap with shapefile CRS/extent. Skipping mask.")
                band_data = src.read(1)
        else:
            # Fallback to reading raw file if no shapefile is used
            band_data = src.read(1)
            
        # Create a spatial index coordinate grid to maintain pixel positions perfectly
        rows, cols = band_data.shape
        row_indices, col_indices = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
        
        # Flatten arrays for 2D tabular structure
        flat_vals = band_data.flatten()
        flat_rows = row_indices.flatten()
        flat_cols = col_indices.flatten()
        
        # Create temporary DataFrame for this time step
        df_temp = pd.DataFrame({
            'Row_Idx': flat_rows,
            'Col_Idx': flat_cols,
            'Year': year,
            'Month': month,
            'LST': flat_vals
        })
        
        # Clean background / NoData values to save massive amounts of memory
        if nodata_val is not None:
            df_temp = df_temp[df_temp['LST'] != nodata_val]
        df_temp = df_temp[~np.isnan(df_temp['LST'])]
        
        # Additional drop for standard common remote sensing nodata placeholders
        df_temp = df_temp[(df_temp['LST'] > -100) & (df_temp['LST'] < 200)] 
        
        all_extracted_records.append(df_temp)
        
    if (idx + 1) % 20 == 0 or (idx + 1) == len(filtered_files):
        print(f"   Processed {idx + 1}/{len(filtered_files)} files...")

# =====================================================================
# STEP 4: CONSOLIDATION & CSV EXPORT
# =====================================================================
print("\nStep 4: Consolidating all data records into master dataframe...")
final_ml_dataset = pd.concat(all_extracted_records, ignore_index=True)

# Generate final file path
csv_filename = "ningxia_growing_season_lst.csv"
final_csv_path = os.path.join(output_dir, csv_filename)

print(f"Saving final structured tabular data to CSV...")
final_ml_dataset.to_csv(final_csv_path, index=False)

print("-" * 60)
print(f"SUCCESS: Extracted dataset ready for Machine Learning!")
print(f"Output Path: {final_csv_path}")
print(f"Total rows extracted: {len(final_ml_dataset):,}")
print(f"Columns generated: {list(final_ml_dataset.columns)}")
print("-" * 60)

Step 1: Scanning directory and applying temporal filtering...
-> Found 140 files matching the growing season criteria.

Step 2 & 3: Aligning spatial grids, masking to Ningxia, and extracting to table...
-> No shapefile provided (or path invalid). Extracting using TIF internal masks.
   Processed 20/140 files...
   Processed 40/140 files...
   Processed 60/140 files...
   Processed 80/140 files...
   Processed 100/140 files...
   Processed 120/140 files...
   Processed 140/140 files...

Step 4: Consolidating all data records into master dataframe...
Saving final structured tabular data to CSV...
------------------------------------------------------------
SUCCESS: Extracted dataset ready for Machine Learning!
Output Path: E:\2_Contribution\Final\Preprocess\1_ML\LST\ningxia_growing_season_lst.csv
Total rows extracted: 7,379,260
Columns generated: ['Row_Idx', 'Col_Idx', 'Year', 'Month', 'LST']
------------------------------------------------------------


In [3]:
import os
import glob
import re
import numpy as np
import pandas as pd
import rasterio

# =====================================================================
# PATH CONFIGURATION
# =====================================================================
input_dir = r"E:\2_Contribution\Final\Preprocess\NDVI"
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML\NDVI"
os.makedirs(output_dir, exist_ok=True)

# =====================================================================
# STEP 1: TEMPORAL FILTERING (Growing Season 2001-2020)
# =====================================================================
print("Step 1: Scanning NDVI directory and applying temporal filtering...")
all_tif_files = glob.glob(os.path.join(input_dir, "*.tif"))
pattern = re.compile(r"NDVI_(\d{4})_(\d{2})\.tif")

filtered_files = []
for filepath in all_tif_files:
    match = pattern.search(os.path.basename(filepath))
    if match:
        year = int(match.group(1))
        month = int(match.group(2))
        # Growing season (Months 4 to 10) between 2001 and 2020
        if 2001 <= year <= 2020 and 4 <= month <= 10:
            filtered_files.append((filepath, year, month))

print(f"-> Found {len(filtered_files)} NDVI files matching the growing season criteria.")

if not filtered_files:
    raise ValueError("No matching NDVI TIF files found. Check your file naming or directory.")

# =====================================================================
# STEP 2 & 3: GRID EXTRACTION & NODATA MASKING
# =====================================================================
print("\nStep 2 & 3: Extracting NDVI pixel values and dropping background masks...")

all_extracted_records = []

for idx, (filepath, year, month) in enumerate(filtered_files):
    with rasterio.open(filepath) as src:
        nodata_val = src.nodata
        band_data = src.read(1)
        
        # Build matching spatial row/col coordinate indices
        rows, cols = band_data.shape
        row_indices, col_indices = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
        
        # Flatten arrays to tabular structure
        flat_vals = band_data.flatten()
        flat_rows = row_indices.flatten()
        flat_cols = col_indices.flatten()
        
        # Create temporary DataFrame
        df_temp = pd.DataFrame({
            'Row_Idx': flat_rows,
            'Col_Idx': flat_cols,
            'Year': year,
            'Month': month,
            'NDVI': flat_vals
        })
        
        # Drop background NoData values
        if nodata_val is not None:
            df_temp = df_temp[df_temp['NDVI'] != nodata_val]
        df_temp = df_temp[~np.isnan(df_temp['NDVI'])]
        
        # Operational NDVI safety bounds filter (-1.0 to 1.0)
        # Note: If your data is scaled by 1000 or 10000, adjust these boundaries accordingly
        df_temp = df_temp[(df_temp['NDVI'] >= -1.0) & (df_temp['NDVI'] <= 1.0)]
        
        all_extracted_records.append(df_temp)
        
    if (idx + 1) % 20 == 0 or (idx + 1) == len(filtered_files):
        print(f"   Processed {idx + 1}/{len(filtered_files)} files...")

# =====================================================================
# STEP 4: CONSOLIDATION & CSV EXPORT
# =====================================================================
print("\nStep 4: Consolidating all NDVI records into master dataframe...")
final_ndvi_dataset = pd.concat(all_extracted_records, ignore_index=True)

csv_filename = "ningxia_growing_season_ndvi.csv"
final_csv_path = os.path.join(output_dir, csv_filename)

print(f"Saving final structured NDVI table to CSV...")
final_ndvi_dataset.to_csv(final_csv_path, index=False)

print("-" * 60)
print(f"SUCCESS: Extracted NDVI dataset ready!")
print(f"Output Path: {final_csv_path}")
print(f"Total rows extracted: {len(final_ndvi_dataset):,}")
print(f"Columns generated: {list(final_ndvi_dataset.columns)}")
print("-" * 60)

Step 1: Scanning NDVI directory and applying temporal filtering...
-> Found 140 NDVI files matching the growing season criteria.

Step 2 & 3: Extracting NDVI pixel values and dropping background masks...
   Processed 20/140 files...
   Processed 40/140 files...
   Processed 60/140 files...
   Processed 80/140 files...
   Processed 100/140 files...
   Processed 120/140 files...
   Processed 140/140 files...

Step 4: Consolidating all NDVI records into master dataframe...
Saving final structured NDVI table to CSV...
------------------------------------------------------------
SUCCESS: Extracted NDVI dataset ready!
Output Path: E:\2_Contribution\Final\Preprocess\1_ML\NDVI\ningxia_growing_season_ndvi.csv
Total rows extracted: 7,379,230
Columns generated: ['Row_Idx', 'Col_Idx', 'Year', 'Month', 'NDVI']
------------------------------------------------------------


In [4]:
import os
import glob
import re
import numpy as np
import pandas as pd
import rasterio

# =====================================================================
# PATH CONFIGURATION
# =====================================================================
input_dir = r"E:\2_Contribution\Final\Preprocess\SMDI"
# Saving directly to the requested master ML folder
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML"
os.makedirs(output_dir, exist_ok=True)

# =====================================================================
# STEP 1: TEMPORAL FILTERING (Growing Season 2001-2020)
# =====================================================================
print("Step 1: Scanning SMDI directory and applying temporal filtering...")
all_tif_files = glob.glob(os.path.join(input_dir, "*.tif"))
pattern = re.compile(r"SMDI_(\d{4})_(\d{2})\.tif")

filtered_files = []
for filepath in all_tif_files:
    match = pattern.search(os.path.basename(filepath))
    if match:
        year = int(match.group(1))
        month = int(match.group(2))
        # Growing season (Months 4 to 10) between 2001 and 2020
        if 2001 <= year <= 2020 and 4 <= month <= 10:
            filtered_files.append((filepath, year, month))

print(f"-> Found {len(filtered_files)} SMDI files matching the growing season criteria.")

if not filtered_files:
    raise ValueError("No matching SMDI TIF files found. Check your file naming or directory.")

# =====================================================================
# STEP 2 & 3: GRID EXTRACTION & NODATA MASKING
# =====================================================================
print("\nStep 2 & 3: Extracting SMDI pixel values and dropping background masks...")

all_extracted_records = []

for idx, (filepath, year, month) in enumerate(filtered_files):
    with rasterio.open(filepath) as src:
        nodata_val = src.nodata
        band_data = src.read(1)
        
        # Build matching spatial row/col coordinate indices
        rows, cols = band_data.shape
        row_indices, col_indices = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
        
        # Flatten arrays to tabular structure
        flat_vals = band_data.flatten()
        flat_rows = row_indices.flatten()
        flat_cols = col_indices.flatten()
        
        # Create temporary DataFrame
        df_temp = pd.DataFrame({
            'Row_Idx': flat_rows,
            'Col_Idx': flat_cols,
            'Year': year,
            'Month': month,
            'SMDI': flat_vals
        })
        
        # Drop background NoData values
        if nodata_val is not None:
            df_temp = df_temp[df_temp['SMDI'] != nodata_val]
        df_temp = df_temp[~np.isnan(df_temp['SMDI'])]
        
        # Operational SMDI index check bounds
        # Typically SMDI ranges from -4 to +4, but we keep an extra safety buffer 
        # to capture extreme drought points while clearing out anomalous values
        df_temp = df_temp[(df_temp['SMDI'] >= -10.0) & (df_temp['SMDI'] <= 10.0)]
        
        all_extracted_records.append(df_temp)
        
    if (idx + 1) % 20 == 0 or (idx + 1) == len(filtered_files):
        print(f"   Processed {idx + 1}/{len(filtered_files)} files...")

# =====================================================================
# STEP 4: CONSOLIDATION & CSV EXPORT
# =====================================================================
print("\nStep 4: Consolidating all SMDI records into master dataframe...")
final_smdi_dataset = pd.concat(all_extracted_records, ignore_index=True)

csv_filename = "ningxia_growing_season_smdi.csv"
final_csv_path = os.path.join(output_dir, csv_filename)

print(f"Saving final structured SMDI table to CSV...")
final_smdi_dataset.to_csv(final_csv_path, index=False)

print("-" * 60)
print(f"SUCCESS: Extracted SMDI dataset ready!")
print(f"Output Path: {final_csv_path}")
print(f"Total rows extracted: {len(final_smdi_dataset):,}")
print(f"Columns generated: {list(final_smdi_dataset.columns)}")
print("-" * 60)

Step 1: Scanning SMDI directory and applying temporal filtering...
-> Found 140 SMDI files matching the growing season criteria.

Step 2 & 3: Extracting SMDI pixel values and dropping background masks...
   Processed 20/140 files...
   Processed 40/140 files...
   Processed 60/140 files...
   Processed 80/140 files...
   Processed 100/140 files...
   Processed 120/140 files...
   Processed 140/140 files...

Step 4: Consolidating all SMDI records into master dataframe...
Saving final structured SMDI table to CSV...
------------------------------------------------------------
SUCCESS: Extracted SMDI dataset ready!
Output Path: E:\2_Contribution\Final\Preprocess\1_ML\ningxia_growing_season_smdi.csv
Total rows extracted: 7,419,160
Columns generated: ['Row_Idx', 'Col_Idx', 'Year', 'Month', 'SMDI']
------------------------------------------------------------


In [6]:
import os
import glob
import re
import numpy as np
import pandas as pd
import rasterio

# =====================================================================
# PATH CONFIGURATION
# =====================================================================
input_dir = r"E:\2_Contribution\Final\Preprocess\SPEI_6"
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML"
os.makedirs(output_dir, exist_ok=True)

# =====================================================================
# STEP 1: TEMPORAL FILTERING (Growing Season 2001-2020)
# =====================================================================
print("Step 1: Scanning SPEI6 directory and applying temporal filtering...")
all_tif_files = glob.glob(os.path.join(input_dir, "*.tif"))
# Regex specifically matches SPEI_6_YYYY_MM.tif
pattern = re.compile(r"SPEI_6_(\d{4})_(\d{2})\.tif")

filtered_files = []
for filepath in all_tif_files:
    match = pattern.search(os.path.basename(filepath))
    if match:
        year = int(match.group(1))
        month = int(match.group(2))
        # Growing season (Months 4 to 10) between 2001 and 2020
        if 2001 <= year <= 2020 and 4 <= month <= 10:
            filtered_files.append((filepath, year, month))

print(f"-> Found {len(filtered_files)} SPEI6 files matching the growing season criteria.")

if not filtered_files:
    raise ValueError("No matching SPEI6 TIF files found. Check your file naming or directory.")

# =====================================================================
# STEP 2 & 3: GRID EXTRACTION & NODATA MASKING
# =====================================================================
print("\nStep 2 & 3: Extracting SPEI6 pixel values and dropping background masks...")

all_extracted_records = []

for idx, (filepath, year, month) in enumerate(filtered_files):
    with rasterio.open(filepath) as src:
        nodata_val = src.nodata
        band_data = src.read(1)
        
        # Build matching spatial row/col coordinate indices
        rows, cols = band_data.shape
        row_indices, col_indices = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
        
        # Flatten arrays to tabular structure
        flat_vals = band_data.flatten()
        flat_rows = row_indices.flatten()
        flat_cols = col_indices.flatten()
        
        # Create temporary DataFrame
        df_temp = pd.DataFrame({
            'Row_Idx': flat_rows,
            'Col_Idx': flat_cols,
            'Year': year,
            'Month': month,
            'SPEI6': flat_vals
        })
        
        # Drop background NoData values
        if nodata_val is not None:
            df_temp = df_temp[df_temp['SPEI6'] != nodata_val]
        df_temp = df_temp[~np.isnan(df_temp['SPEI6'])]
        
        # Operational SPEI range check bounds
        # Standardized indices usually range from -3 to +3, keeping a safe window
        df_temp = df_temp[(df_temp['SPEI6'] >= -10.0) & (df_temp['SPEI6'] <= 10.0)]
        
        all_extracted_records.append(df_temp)
        
    if (idx + 1) % 20 == 0 or (idx + 1) == len(filtered_files):
        print(f"   Processed {idx + 1}/{len(filtered_files)} files...")

# =====================================================================
# STEP 4: CONSOLIDATION & CSV EXPORT
# =====================================================================
print("\nStep 4: Consolidating all SPEI6 records into master dataframe...")
final_spei_dataset = pd.concat(all_extracted_records, ignore_index=True)

csv_filename = "ningxia_growing_season_spei6.csv"
final_csv_path = os.path.join(output_dir, csv_filename)

print(f"Saving final structured SPEI6 table to CSV...")
final_spei_dataset.to_csv(final_csv_path, index=False)

print("-" * 60)
print(f"SUCCESS: Extracted SPEI6 dataset ready!")
print(f"Output Path: {final_csv_path}")
print(f"Total rows extracted: {len(final_spei_dataset):,}")
print(f"Columns generated: {list(final_spei_dataset.columns)}")
print("-" * 60)

Step 1: Scanning SPEI6 directory and applying temporal filtering...
-> Found 140 SPEI6 files matching the growing season criteria.

Step 2 & 3: Extracting SPEI6 pixel values and dropping background masks...
   Processed 20/140 files...
   Processed 40/140 files...
   Processed 60/140 files...
   Processed 80/140 files...
   Processed 100/140 files...
   Processed 120/140 files...
   Processed 140/140 files...

Step 4: Consolidating all SPEI6 records into master dataframe...
Saving final structured SPEI6 table to CSV...
------------------------------------------------------------
SUCCESS: Extracted SPEI6 dataset ready!
Output Path: E:\2_Contribution\Final\Preprocess\1_ML\ningxia_growing_season_spei6.csv
Total rows extracted: 7,416,220
Columns generated: ['Row_Idx', 'Col_Idx', 'Year', 'Month', 'SPEI6']
------------------------------------------------------------


In [7]:
import os
import glob
import re
import numpy as np
import pandas as pd
import rasterio

# =====================================================================
# PATH CONFIGURATION
# =====================================================================
input_dir = r"E:\2_Contribution\Final\Preprocess\TVDI"
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML"
os.makedirs(output_dir, exist_ok=True)

# =====================================================================
# STEP 1: TEMPORAL FILTERING (Growing Season 2001-2020)
# =====================================================================
print("Step 1: Scanning TVDI directory and applying temporal filtering...")
all_tif_files = glob.glob(os.path.join(input_dir, "*.tif"))
# Regex specifically matches TVDI_YYYY_MM.tif
pattern = re.compile(r"TVDI_(\d{4})_(\d{2})\.tif")

filtered_files = []
for filepath in all_tif_files:
    match = pattern.search(os.path.basename(filepath))
    if match:
        year = int(match.group(1))
        month = int(match.group(2))
        # Growing season (Months 4 to 10) between 2001 and 2020
        if 2001 <= year <= 2020 and 4 <= month <= 10:
            filtered_files.append((filepath, year, month))

print(f"-> Found {len(filtered_files)} TVDI files matching the growing season criteria.")

if not filtered_files:
    raise ValueError("No matching TVDI TIF files found. Check your file naming or directory.")

# =====================================================================
# STEP 2 & 3: GRID EXTRACTION & NODATA MASKING
# =====================================================================
print("\nStep 2 & 3: Extracting TVDI pixel values and dropping background masks...")

all_extracted_records = []

for idx, (filepath, year, month) in enumerate(filtered_files):
    with rasterio.open(filepath) as src:
        nodata_val = src.nodata
        band_data = src.read(1)
        
        # Build matching spatial row/col coordinate indices
        rows, cols = band_data.shape
        row_indices, col_indices = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
        
        # Flatten arrays to tabular structure
        flat_vals = band_data.flatten()
        flat_rows = row_indices.flatten()
        flat_cols = col_indices.flatten()
        
        # Create temporary DataFrame
        df_temp = pd.DataFrame({
            'Row_Idx': flat_rows,
            'Col_Idx': flat_cols,
            'Year': year,
            'Month': month,
            'TVDI': flat_vals
        })
        
        # Drop background NoData values
        if nodata_val is not None:
            df_temp = df_temp[df_temp['TVDI'] != nodata_val]
        df_temp = df_temp[~np.isnan(df_temp['TVDI'])]
        
        # Operational TVDI range check bounds
        # TVDI values strictly range between 0 (wet edge) and 1 (dry edge)
        df_temp = df_temp[(df_temp['TVDI'] >= 0.0) & (df_temp['TVDI'] <= 1.0)]
        
        all_extracted_records.append(df_temp)
        
    if (idx + 1) % 20 == 0 or (idx + 1) == len(filtered_files):
        print(f"   Processed {idx + 1}/{len(filtered_files)} files...")

# =====================================================================
# STEP 4: CONSOLIDATION & CSV EXPORT
# =====================================================================
print("\nStep 4: Consolidating all TVDI records into master dataframe...")
final_tvdi_dataset = pd.concat(all_extracted_records, ignore_index=True)

csv_filename = "ningxia_growing_season_tvdi.csv"
final_csv_path = os.path.join(output_dir, csv_filename)

print(f"Saving final structured TVDI table to CSV...")
final_tvdi_dataset.to_csv(final_csv_path, index=False)

print("-" * 60)
print(f"SUCCESS: Extracted TVDI dataset ready!")
print(f"Output Path: {final_csv_path}")
print(f"Total rows extracted: {len(final_tvdi_dataset):,}")
print(f"Columns generated: {list(final_tvdi_dataset.columns)}")
print("-" * 60)

Step 1: Scanning TVDI directory and applying temporal filtering...
-> Found 140 TVDI files matching the growing season criteria.

Step 2 & 3: Extracting TVDI pixel values and dropping background masks...
   Processed 20/140 files...
   Processed 40/140 files...
   Processed 60/140 files...
   Processed 80/140 files...
   Processed 100/140 files...
   Processed 120/140 files...
   Processed 140/140 files...

Step 4: Consolidating all TVDI records into master dataframe...
Saving final structured TVDI table to CSV...
------------------------------------------------------------
SUCCESS: Extracted TVDI dataset ready!
Output Path: E:\2_Contribution\Final\Preprocess\1_ML\ningxia_growing_season_tvdi.csv
Total rows extracted: 7,348,019
Columns generated: ['Row_Idx', 'Col_Idx', 'Year', 'Month', 'TVDI']
------------------------------------------------------------


In [8]:
import os
import glob
import re
import numpy as np
import pandas as pd
import rasterio

# =====================================================================
# PATH CONFIGURATION
# =====================================================================
input_dir = r"E:\2_Contribution\Final\Preprocess\VCI"
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML"
os.makedirs(output_dir, exist_ok=True)

# =====================================================================
# STEP 1: TEMPORAL FILTERING (Growing Season 2001-2020)
# =====================================================================
print("Step 1: Scanning VCI directory and applying temporal filtering...")
all_tif_files = glob.glob(os.path.join(input_dir, "*.tif"))
# Regex specifically matches VCI_YYYY_MM.tif
pattern = re.compile(r"VCI_(\d{4})_(\d{2})\.tif")

filtered_files = []
for filepath in all_tif_files:
    match = pattern.search(os.path.basename(filepath))
    if match:
        year = int(match.group(1))
        month = int(match.group(2))
        # Growing season (Months 4 to 10) between 2001 and 2020
        if 2001 <= year <= 2020 and 4 <= month <= 10:
            filtered_files.append((filepath, year, month))

print(f"-> Found {len(filtered_files)} VCI files matching the growing season criteria.")

if not filtered_files:
    raise ValueError("No matching VCI TIF files found. Check your file naming or directory.")

# =====================================================================
# STEP 2 & 3: GRID EXTRACTION & NODATA MASKING
# =====================================================================
print("\nStep 2 & 3: Extracting VCI pixel values and dropping background masks...")

all_extracted_records = []

for idx, (filepath, year, month) in enumerate(filtered_files):
    with rasterio.open(filepath) as src:
        nodata_val = src.nodata
        band_data = src.read(1)
        
        # Build matching spatial row/col coordinate indices
        rows, cols = band_data.shape
        row_indices, col_indices = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')
        
        # Flatten arrays to tabular structure
        flat_vals = band_data.flatten()
        flat_rows = row_indices.flatten()
        flat_cols = col_indices.flatten()
        
        # Create temporary DataFrame
        df_temp = pd.DataFrame({
            'Row_Idx': flat_rows,
            'Col_Idx': flat_cols,
            'Year': year,
            'Month': month,
            'VCI': flat_vals
        })
        
        # Drop background NoData values
        if nodata_val is not None:
            df_temp = df_temp[df_temp['VCI'] != nodata_val]
        df_temp = df_temp[~np.isnan(df_temp['VCI'])]
        
        # Operational VCI range check bounds
        # VCI typically ranges from 0 to 100 (%) or 0.0 to 1.0. 
        # Keeping a broad safety window to fit your data scale while removing garbage.
        df_temp = df_temp[(df_temp['VCI'] >= 0.0) & (df_temp['VCI'] <= 100.0)]
        
        all_extracted_records.append(df_temp)
        
    if (idx + 1) % 20 == 0 or (idx + 1) == len(filtered_files):
        print(f"   Processed {idx + 1}/{len(filtered_files)} files...")

# =====================================================================
# STEP 4: CONSOLIDATION & CSV EXPORT
# =====================================================================
print("\nStep 4: Consolidating all VCI records into master dataframe...")
final_vci_dataset = pd.concat(all_extracted_records, ignore_index=True)

csv_filename = "ningxia_growing_season_vci.csv"
final_csv_path = os.path.join(output_dir, csv_filename)

print(f"Saving final structured VCI table to CSV...")
final_vci_dataset.to_csv(final_csv_path, index=False)

print("-" * 60)
print(f"SUCCESS: Extracted VCI dataset ready!")
print(f"Output Path: {final_csv_path}")
print(f"Total rows extracted: {len(final_vci_dataset):,}")
print(f"Columns generated: {list(final_vci_dataset.columns)}")
print("-" * 60)

Step 1: Scanning VCI directory and applying temporal filtering...
-> Found 140 VCI files matching the growing season criteria.

Step 2 & 3: Extracting VCI pixel values and dropping background masks...
   Processed 20/140 files...
   Processed 40/140 files...
   Processed 60/140 files...
   Processed 80/140 files...
   Processed 100/140 files...
   Processed 120/140 files...
   Processed 140/140 files...

Step 4: Consolidating all VCI records into master dataframe...
Saving final structured VCI table to CSV...
------------------------------------------------------------
SUCCESS: Extracted VCI dataset ready!
Output Path: E:\2_Contribution\Final\Preprocess\1_ML\ningxia_growing_season_vci.csv
Total rows extracted: 7,418,320
Columns generated: ['Row_Idx', 'Col_Idx', 'Year', 'Month', 'VCI']
------------------------------------------------------------


In [10]:
import os
import pandas as pd

# =====================================================================
# PATH CONFIGURATION
# =====================================================================
ml_dir = r"E:\2_Contribution\Final\Preprocess\1_ML"

# Dictionary defining where each of your processed CSV files is located
# Adjust the filenames if they differ slightly on your drive
file_paths = {
    'LST': os.path.join(ml_dir, r"ningxia_growing_season_lst.csv"),
    'NDVI': os.path.join(ml_dir, r"ningxia_growing_season_ndvi.csv"),
    'SMDI': os.path.join(ml_dir, "ningxia_growing_season_smdi.csv"),
    'SPEI3': os.path.join(ml_dir, "ningxia_growing_season_spei3.csv"),  # Assuming naming pattern
    'SPEI6': os.path.join(ml_dir, "ningxia_growing_season_spei6.csv"),
    'TVDI': os.path.join(ml_dir, "ningxia_growing_season_tvdi.csv"),    # Assuming naming pattern
    'VCI': os.path.join(ml_dir, "ningxia_growing_season_vci.csv")
}

# Shared coordinate keys used to align data points across space and time
merge_keys = ['Row_Idx', 'Col_Idx', 'Year', 'Month']

# =====================================================================
# STEP 2: SEQUENTIAL RELATIONAL MERGE (INNER JOIN)
# =====================================================================
master_df = None

print("Starting verification and alignment of all drought index tables...")

for index_name, path in file_paths.items():
    if not os.path.exists(path):
        print(f"Warning: {index_name} file not found at {path}. Skipping this index.")
        continue
        
    print(f"-> Loading and aligning {index_name} dataset...")
    df_temp = pd.read_csv(path)
    
    if master_df is None:
        # Initialize the master dataframe with the first index (e.g., LST)
        master_df = df_temp
    else:
        # Inner merge guarantees only perfectly matching rows are kept.
        # This completely resolves the VCI / TVDI alignment issues.
        master_df = pd.merge(master_df, df_temp, on=merge_keys, how='inner')
        print(f"   Current aligned row count: {len(master_df):,}")

# =====================================================================
# EXPORT FINAL UNIFIED DATASET
# =====================================================================
if master_df is not None:
    output_csv_path = os.path.join(ml_dir, "unified_growing_season_dataset.csv")
    
    print("\nSaving unified dataset to disk...")
    master_df.to_csv(output_csv_path, index=False)
    
    print("-" * 60)
    print("SUCCESS: Unified Machine Learning Dataset Created!")
    print(f"Destination: {output_csv_path}")
    print(f"Total matching observations (Rows): {len(master_df):,}")
    print(f"Features included (Columns): {list(master_df.columns)}")
    print("-" * 60)
else:
    print("Error: No datasets were found or merged. Check file paths.")

Starting verification and alignment of all drought index tables...
-> Loading and aligning LST dataset...
-> Loading and aligning NDVI dataset...
   Current aligned row count: 7,379,230
-> Loading and aligning SMDI dataset...
   Current aligned row count: 7,379,230
-> Loading and aligning SPEI3 dataset...
   Current aligned row count: 7,376,430
-> Loading and aligning SPEI6 dataset...
   Current aligned row count: 7,376,430
-> Loading and aligning TVDI dataset...
   Current aligned row count: 7,323,209
-> Loading and aligning VCI dataset...
   Current aligned row count: 7,323,209

Saving unified dataset to disk...
------------------------------------------------------------
SUCCESS: Unified Machine Learning Dataset Created!
Destination: E:\2_Contribution\Final\Preprocess\1_ML\unified_growing_season_dataset.csv
Total matching observations (Rows): 7,323,209
Features included (Columns): ['Row_Idx', 'Col_Idx', 'Year', 'Month', 'LST', 'NDVI', 'SMDI', 'SPEI_3', 'SPEI6', 'TVDI', 'VCI']
------

In [11]:
#Preprocessing csv file data 
# =====================================================================
# STEP 1: IMPORT NECESSARY LIBRARIES
# =====================================================================
print("Step 1: Importing necessary libraries...")
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# Define paths
ml_dir = r"E:\2_Contribution\Final\Preprocess\1_ML"
input_csv = os.path.join(ml_dir, "unified_growing_season_dataset.csv")
output_csv = os.path.join(ml_dir, "fully_preprocessed_ml_ready.csv")

# =====================================================================
# STEP 2: READ DATASET
# =====================================================================
print("\nStep 2: Reading dataset...")
if not os.path.exists(input_csv):
    raise FileNotFoundError(f"Could not find the unified dataset at {input_csv}. Please run the merge script first.")

df = pd.read_csv(input_csv)
print(f"Loaded dataset successfully. Shape: {df.shape[0]:,} rows, {df.shape[1]} columns.")

# =====================================================================
# STEP 3: SANITY CHECK OF DATA
# =====================================================================
print("\nStep 3: Conducting sanity check of data...")
print("-" * 50)
print("Data structural overview:")
print(df.info())
print("\nMissing values count per column:")
print(df.isnull().sum())
print("\nStatistical distribution baseline:")
print(df.describe().T)
print("-" * 50)

# =====================================================================
# STEP 4: EXPLORATORY DATA ANALYSIS (EDA)
# =====================================================================
print("\nStep 4: Running Exploratory Data Analysis (EDA)...")
# Calculate Pearson correlation matrix for all environmental variables
feature_cols = [col for col in df.columns if col not in ['Row_Idx', 'Col_Idx', 'Year', 'Month']]
corr_matrix = df[feature_cols].corr()

# Save the correlation heatmap for your journal paper
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Matrix of Drought Indices & Features (Ningxia)", fontsize=14, pad=15)
plt.tight_layout()
plot_path = os.path.join(ml_dir, "drought_indices_correlation_matrix.png")
plt.savefig(plot_path, dpi=300)
plt.close()
print(f"-> Correlation matrix heatmap saved successfully to:\n   {plot_path}")

# =====================================================================
# STEP 5: MISSING VALUE TREATMENTS
# =====================================================================
print("\nStep 5: Treating missing values...")
# Because we performed an inner join, explicit missing entries should be minimal/zero.
# However, as a safety protocol, we drop any rows containing NaN values.
initial_rows = len(df)
df = df.dropna(subset=feature_cols)
print(f"-> Removed {initial_rows - len(df):,} rows containing missing/null observations.")

# =====================================================================
# STEP 6: OUTLIERS TREATMENTS
# =====================================================================
print("\nStep 6: Treating outliers using Interquartile Range (IQR)...")
# We apply outlier filtering strictly to continuous environmental metrics
outlier_targets = [col for col in feature_cols if col != 'VCI']  # Keep VCI target intact

before_outlier_count = len(df)
for col in outlier_targets:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Filter rows within the acceptable theoretical bounds
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

print(f"-> Filtered out {before_outlier_count - len(df):,} outlier noise rows across feature columns.")

# =====================================================================
# STEP 7: DUPLICATES & GARBAGE VALUE TREATMENTS
# =====================================================================
print("\nStep 7: Treating duplicate records and clearing invalid garbage bounds...")
# 1. Remove strict duplicate lines
before_dup_count = len(df)
df = df.drop_duplicates()
print(f"-> Dropped {before_dup_count - len(df):,} exact duplicate rows.")

# 2. Eliminate any remaining generic background garbage values outside operational ranges
# For example, filtering out common fill values like -9999 or 9999 if they bypassed early checks
if 'LST' in df.columns:
    df = df[(df['LST'] > -50) & (df['LST'] < 100)]
if 'NDVI' in df.columns:
    df = df[(df['NDVI'] >= -1.0) & (df['NDVI'] <= 1.0)]

# =====================================================================
# STEP 8: NORMALIZATION
# =====================================================================
print("\nStep 8: Applying MinMax Normalization to all feature sets...")
# Create scalers for features. We scale SMDI, SPEI, etc., alongside LST/NDVI.
scaler_features = MinMaxScaler()

# We normalize the features but keep 'VCI' in its original scale since it is your target variable.
# (If you want to scale the target VCI too, just change this list to include it)
features_to_scale = [col for col in feature_cols if col != 'VCI']

# Apply MinMax Scaling [0, 1]
df[features_to_scale] = scaler_features.fit_transform(df[features_to_scale])
print(f"-> Scaled columns to [0, 1] range: {features_to_scale}")

# =====================================================================
# STEP 9: ENCODING OF DATA
# =====================================================================
print("\nStep 9: Encoding cyclic temporal attributes...")
# Since 'Month' is a categorical/temporal factor representing the growing season cycle (4-10),
# we apply One-Hot Encoding so the machine learning model can understand seasonal characteristics.
df = pd.get_dummies(df, columns=['Month'], drop_first=True, dtype=int)
print(f"-> Encoded 'Month' category columns. Current dataset features: {list(df.columns)}")

# =====================================================================
# SAVE COMPLETED OUTPUT DATASET
# =====================================================================
print("\nSaving finalized preprocessed dataset to disk...")
df.to_csv(output_csv, index=False)

print("-" * 60)
print("SUCCESS: Full Preprocessing Pipeline Finished Safely!")
print(f"Output File Path: {output_csv}")
print(f"Final Data Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("-" * 60)

Step 1: Importing necessary libraries...

Step 2: Reading dataset...
Loaded dataset successfully. Shape: 7,323,209 rows, 11 columns.

Step 3: Conducting sanity check of data...
--------------------------------------------------
Data structural overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7323209 entries, 0 to 7323208
Data columns (total 11 columns):
 #   Column   Dtype  
---  ------   -----  
 0   Row_Idx  int64  
 1   Col_Idx  int64  
 2   Year     int64  
 3   Month    int64  
 4   LST      float64
 5   NDVI     float64
 6   SMDI     float64
 7   SPEI_3   float64
 8   SPEI6    float64
 9   TVDI     float64
 10  VCI      float64
dtypes: float64(7), int64(4)
memory usage: 614.6 MB
None

Missing values count per column:
Row_Idx    0
Col_Idx    0
Year       0
Month      0
LST        0
NDVI       0
SMDI       0
SPEI_3     0
SPEI6      0
TVDI       0
VCI        0
dtype: int64

Statistical distribution baseline:
             count         mean         std          min        

In [14]:
# Spearman's Rank Correlation Matrix

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# CONFIGURATION & CONFIG PATHS
# =====================================================================
ml_dir = r"E:\2_Contribution\Final\Preprocess\1_ML"
input_csv = os.path.join(ml_dir, "fully_preprocessed_ml_ready.csv")
output_plot = os.path.join(ml_dir, "spearman_correlation_heatmap.png")

# Set publication style configurations
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

# Load dataset
if not os.path.exists(input_csv):
    raise FileNotFoundError(f"Missing preprocessed dataset at: {input_csv}")

df = pd.read_csv(input_csv)

# Isolate the continuous environmental variables (Exclude Row/Col/Year and encoded Month columns)
core_features = ['LST', 'NDVI', 'SMDI', 'SPEI_3', 'SPEI6', 'TVDI', 'VCI']
df_features = df[core_features]

# =====================================================================
# COMPUTATION: SPEARMAN'S RANK CORRELATION
# =====================================================================
print("Computing Spearman's Rank Correlation Matrix...")
spearman_corr = df_features.corr(method='spearman')

# Generate a mask for the upper triangle to create a clean, elegant layout
mask = np.triu(np.ones_like(spearman_corr, dtype=bool))

# =====================================================================
# PLOTTING: HIGH-RESOLUTION ACADEMIC HEATMAP
# =====================================================================
fig, ax = plt.subplots(figsize=(10, 8.5))

# Diverging color palette tailored for climate/drought manuscripts
cmap = sns.diverging_palette(230, 20, as_cmap=True)

print("Generating publication-ready layout...")
sns.heatmap(
    spearman_corr, 
    mask=mask, 
    cmap=cmap, 
    vmax=1.0, 
    vmin=-1.0, 
    center=0,
    annot=True, 
    fmt=".2f", 
    square=True, 
    linewidths=.8, 
    cbar_kws={"shrink": .75, "label": "Spearman's Rank Correlation Coefficient ($\\rho$)"},
    annot_kws={"size": 11, "weight": "medium"},
    ax=ax
)

# Refine labels and axis typography
ax.set_title("Spearman Rank Correlation Matrix of Drought Indices (Ningxia)", fontsize=14, pad=20, weight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right', fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)

# Clean up layout adjustments
plt.tight_layout()

# Save image at 300 DPI for high-print journal specifications
plt.savefig(output_plot, dpi=300, bbox_inches='tight')
plt.close()

print("-" * 60)
print(f"SUCCESS: High-resolution Spearman heatmap saved to:\n{output_plot}")
print("-" * 60)

Computing Spearman's Rank Correlation Matrix...
Generating publication-ready layout...
------------------------------------------------------------
SUCCESS: High-resolution Spearman heatmap saved to:
E:\2_Contribution\Final\Preprocess\1_ML\spearman_correlation_heatmap.png
------------------------------------------------------------


In [15]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# CONFIGURATION & FILE PATHS
# =====================================================================
ml_dir = r"E:\2_Contribution\Final\Preprocess\1_ML"
input_csv = os.path.join(ml_dir, "fully_preprocessed_ml_ready.csv")
output_plot = os.path.join(ml_dir, "spearman_heatmap_ndvi_target.png")

# Set publication style configurations (Serif font family)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

# Load preprocessed dataset
if not os.path.exists(input_csv):
    raise FileNotFoundError(f"Missing preprocessed dataset at: {input_csv}")

df = pd.read_csv(input_csv)

# Re-order features so your new target (NDVI) sits at the very bottom row
# This makes it visually immediate for reviewers to read the target's correlations
core_features = ['LST', 'SMDI', 'SPEI_3', 'SPEI6', 'TVDI', 'VCI', 'NDVI']
df_features = df[core_features]

# =====================================================================
# COMPUTATION: SPEARMAN'S RANK CORRELATION
# =====================================================================
print("Computing Spearman's Rank Correlation Matrix...")
spearman_corr = df_features.corr(method='spearman')

# Generate a mask for the upper triangle for a clean, elegant layout
mask = np.triu(np.ones_like(spearman_corr, dtype=bool))

# =====================================================================
# PLOTTING: HIGH-RESOLUTION ACADEMIC HEATMAP
# =====================================================================
fig, ax = plt.subplots(figsize=(10, 8.5))

# Balanced diverging color palette for climate manuscripts (Blue = Negative, Red = Positive)
cmap = sns.diverging_palette(230, 20, as_cmap=True)

print("Generating publication-ready layout...")
sns.heatmap(
    spearman_corr, 
    mask=mask, 
    cmap=cmap, 
    vmax=1.0, 
    vmin=-1.0, 
    center=0,
    annot=True, 
    fmt=".2f", 
    square=True, 
    linewidths=.8, 
    cbar_kws={"shrink": .75, "label": "Spearman's Rank Correlation Coefficient ($\\rho$)"},
    annot_kws={"size": 11, "weight": "medium"},
    ax=ax
)

# Refine labels and axis typography
ax.set_title("Spearman Rank Correlation Matrix with NDVI Target (Ningxia)", fontsize=14, pad=20, weight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right', fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)

# Clean up layout adjustments and save at 300 DPI
plt.tight_layout()
plt.savefig(output_plot, dpi=300, bbox_inches='tight')
plt.close()

print("-" * 60)
print(f"SUCCESS: High-resolution Spearman heatmap saved to:\n{output_plot}")
print("-" * 60)

Computing Spearman's Rank Correlation Matrix...
Generating publication-ready layout...
------------------------------------------------------------
SUCCESS: High-resolution Spearman heatmap saved to:
E:\2_Contribution\Final\Preprocess\1_ML\spearman_heatmap_ndvi_target.png
------------------------------------------------------------


In [19]:
# =====================================================================
# STEP 1: IMPORT NECESSARY LIBRARIES
# =====================================================================
print("Step 1: Importing necessary libraries...")
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# Define paths
ml_dir = r"E:\2_Contribution\Final\Preprocess\1_ML"
input_csv = os.path.join(ml_dir, "unified_growing_season_dataset.csv")
output_csv = os.path.join(ml_dir, "1stfully_preprocessed_lagged_ml_ready.csv")

# Set font family for publication-standard figures
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

# =====================================================================
# STEP 2: READ DATASET
# =====================================================================
print("\nStep 2: Reading dataset...")
if not os.path.exists(input_csv):
    raise FileNotFoundError(f"Could not find the unified dataset at {input_csv}.")

df = pd.read_csv(input_csv)
print(f"Loaded dataset successfully. Baseline Shape: {df.shape[0]:,} rows.")

# =====================================================================
# ADVANCED FEATURE ENGINEERING: STEP-LAG RESPONSES
# =====================================================================
print("\nStep 2.5: Generating 1-month and 2-month spatial-temporal feature lags...")

# Sort data explicitly by space and time to ensure rolling shifts target matching records
df = df.sort_values(by=['Row_Idx', 'Col_Idx', 'Year', 'Month']).reset_index(drop=True)

# Define variables that exhibit dynamic ecological lag responses
lag_features = ['LST', 'SMDI', 'SPEI_3', 'SPEI6', 'TVDI']

# Group by individual pixel coordinates to prevent geographic data bleeding
grouped = df.groupby(['Row_Idx', 'Col_Idx'])

for col in lag_features:
    # 1-Month Lag
    df[f'{col}_lag1'] = grouped[col].shift(1)
    # 2-Month Lag
    df[f'{col}_lag2'] = grouped[col].shift(2)

print("-> Time-lags engineered. Removing rolling edge artifacts...")
# Shifting creates NaN values for the first 2 months of each pixel's history. We drop them.
df = df.dropna().reset_index(drop=True)
print(f"   Shape after clearing lag edge entries: {df.shape[0]:,} rows.")

# =====================================================================
# STEP 3: SANITY CHECK OF DATA
# =====================================================================
print("\nStep 3: Conducting sanity check of lagged data structure...")
print(f"Total features now available: {df.shape[1]}")
print(f"Checking for any generated NaN values: {df.isnull().sum().sum()}")

# =====================================================================
# STEP 4: EXPLORATORY DATA ANALYSIS (EDA) - SPEARMAN LAGGED MATRIX
# =====================================================================
print("\nStep 4: Plotting professional Spearman Heatmap with Lagged Features...")

# Update core feature list to include original features + newly generated lags + target VCI
all_numeric_features = ['LST', 'LST_lag1', 'LST_lag2', 
                        'SMDI', 'SMDI_lag1', 'SMDI_lag2', 
                        'SPEI_3', 'SPEI_3_lag1', 'SPEI_3_lag2', 
                        'SPEI6', 'SPEI6_lag1', 'SPEI6_lag2', 
                        'TVDI', 'TVDI_lag1', 'TVDI_lag2', 
                        'NDVI', 'VCI']

spearman_corr = df[all_numeric_features].corr(method='spearman')
mask = np.triu(np.ones_like(spearman_corr, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 12))
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(
    spearman_corr, mask=mask, cmap=cmap, vmax=1.0, vmin=-1.0, center=0,
    annot=True, fmt=".2f", square=True, linewidths=.5,
    cbar_kws={"shrink": .7, "label": "Spearman's Rank Correlation Coefficient ($\\rho$)"},
    annot_kws={"size": 8}, ax=ax
)

ax.set_title("Spearman Rank Correlation Matrix with Lagged Drivers (Ningxia)", fontsize=14, pad=20, weight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()

plot_path = os.path.join(ml_dir, "1stspearman_lagged_heatmap.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"-> Lagged heatmap exported to: {plot_path}")

# =====================================================================
# STEP 5 & 7: MISSING, DUPLICATES & GARBAGE VALUE TREATMENTS
# =====================================================================
print("\nStep 5 & 7: Cleaning up duplicate elements and boundary noise...")
df = df.drop_duplicates()

# =====================================================================
# STEP 6: OUTLIERS TREATMENTS
# =====================================================================
print("\nStep 6: Treating out-of-bounds outliers using IQR...")
continuous_predictors = [col for col in all_numeric_features if col != 'VCI']

for col in continuous_predictors:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

print(f"-> Final row count after outlier filter: {df.shape[0]:,} rows.")

# =====================================================================
# STEP 8: NORMALIZATION
# =====================================================================
print("\nStep 8: Applying MinMax Normalization to all features...")
scaler = MinMaxScaler()
# Normalize everything except our baseline keys and the target VCI variable
features_to_scale = [col for col in df.columns if col not in ['Row_Idx', 'Col_Idx', 'Year', 'Month', 'VCI']]

df[features_to_scale] = scaler.fit_transform(df[features_to_scale])
print(f"-> Successfully normalized {len(features_to_scale)} feature inputs.")

# =====================================================================
# STEP 9: ENCODING OF DATA
# =====================================================================
print("\nStep 9: Encoding cyclic Month parameters...")
df = pd.get_dummies(df, columns=['Month'], drop_first=True, dtype=int)

# =====================================================================
# EXPORT FINAL DATASET
# =====================================================================
print("\nSaving finalized model-ready dataset with lags...")
df.to_csv(output_csv, index=False)

print("-" * 60)
print("SUCCESS: Advanced Lagged Preprocessing Complete!")
print(f"Final Dataset Location: {output_csv}")
print(f"Matrix Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("-" * 60)

Step 1: Importing necessary libraries...

Step 2: Reading dataset...
Loaded dataset successfully. Baseline Shape: 7,323,209 rows.

Step 2.5: Generating 1-month and 2-month spatial-temporal feature lags...
-> Time-lags engineered. Removing rolling edge artifacts...
   Shape after clearing lag edge entries: 7,218,591 rows.

Step 3: Conducting sanity check of lagged data structure...
Total features now available: 21
Checking for any generated NaN values: 0

Step 4: Plotting professional Spearman Heatmap with Lagged Features...
-> Lagged heatmap exported to: E:\2_Contribution\Final\Preprocess\1_ML\1stspearman_lagged_heatmap.png

Step 5 & 7: Cleaning up duplicate elements and boundary noise...

Step 6: Treating out-of-bounds outliers using IQR...
-> Final row count after outlier filter: 6,670,171 rows.

Step 8: Applying MinMax Normalization to all features...
-> Successfully normalized 16 feature inputs.

Step 9: Encoding cyclic Month parameters...

Saving finalized model-ready dataset with

In [17]:
# =====================================================================
# PART 1: DATA PREPROCESSING, FEATURE ENGINEERING & SPEARMAN ANALYSIS
# =====================================================================

print("=" * 70)
print("ADVANCED PREPROCESSING PIPELINE - PART 1")
print("=" * 70)

# =====================================================================
# STEP 1: IMPORT LIBRARIES
# =====================================================================
print("\nStep 1: Importing libraries...")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from sklearn.preprocessing import MinMaxScaler

# =====================================================================
# DEFINE PATHS
# =====================================================================
ml_dir = r"E:\2_Contribution\Final\Preprocess\1_ML"

input_csv = os.path.join(
    ml_dir,
    "unified_growing_season_dataset.csv"
)

# Publication Font
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"] + plt.rcParams["font.serif"]

# =====================================================================
# STEP 2: LOAD DATA
# =====================================================================
print("\nStep 2: Reading dataset...")

if not os.path.exists(input_csv):
    raise FileNotFoundError(input_csv)

df = pd.read_csv(input_csv)

print(f"Dataset Loaded Successfully")
print(f"Rows : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

# =====================================================================
# STEP 3: SORT DATA
# =====================================================================
print("\nStep 3: Sorting pixel time-series...")

df = (
    df
    .sort_values(
        by=["Row_Idx", "Col_Idx", "Year", "Month"]
    )
    .reset_index(drop=True)
)

# =====================================================================
# STEP 4: CREATE TEMPORAL LAG FEATURES
# =====================================================================
print("\nStep 4: Generating lagged variables...")

lag_features = [
    "LST",
    "SMDI",
    "SPEI_3",
    "SPEI6",
    "TVDI"
]

grouped = df.groupby(["Row_Idx", "Col_Idx"])

for feature in lag_features:

    df[f"{feature}_lag1"] = grouped[feature].shift(1)

    df[f"{feature}_lag2"] = grouped[feature].shift(2)

print("Lag variables generated successfully.")

# =====================================================================
# STEP 5: REMOVE LAG NaNs
# =====================================================================
print("\nStep 5: Removing lag edge NaNs...")

before = len(df)

df = (
    df
    .dropna()
    .reset_index(drop=True)
)

after = len(df)

print(f"Removed : {before-after:,} rows")
print(f"Remaining : {after:,}")

# =====================================================================
# STEP 6: REMOVE DUPLICATES
# =====================================================================
print("\nStep 6: Removing duplicated observations...")

before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print(f"Removed : {before-after:,} duplicate rows")

# =====================================================================
# STEP 7: OUTLIER REMOVAL (IQR - SINGLE BOOLEAN MASK)
# =====================================================================
print("\nStep 7: Removing outliers using IQR...")

predictor_columns = [

    "LST","LST_lag1","LST_lag2",

    "SMDI","SMDI_lag1","SMDI_lag2",

    "SPEI_3","SPEI_3_lag1","SPEI_3_lag2",

    "SPEI6","SPEI6_lag1","SPEI6_lag2",

    "TVDI","TVDI_lag1","TVDI_lag2",

    "NDVI"

]

valid_mask = pd.Series(True, index=df.index)

for col in predictor_columns:

    Q1 = df[col].quantile(0.25)

    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR

    upper = Q3 + 1.5 * IQR

    valid_mask &= df[col].between(lower, upper)

before = len(df)

df = (
    df.loc[valid_mask]
      .reset_index(drop=True)
)

after = len(df)

print(f"Rows removed : {before-after:,}")
print(f"Rows remaining : {after:,}")

# =====================================================================
# STEP 8: SANITY CHECK
# =====================================================================
print("\nStep 8: Dataset sanity check...")

print(f"Total Missing Values : {df.isna().sum().sum()}")
print(f"Duplicate Rows       : {df.duplicated().sum()}")

# =====================================================================
# STEP 9: SPEARMAN CORRELATION
# =====================================================================
print("\nStep 9: Computing Spearman correlation matrix...")

corr_features = [

    "LST","LST_lag1","LST_lag2",

    "SMDI","SMDI_lag1","SMDI_lag2",

    "SPEI_3","SPEI_3_lag1","SPEI_3_lag2",

    "SPEI6","SPEI6_lag1","SPEI6_lag2",

    "TVDI","TVDI_lag1","TVDI_lag2",

    "NDVI",

    "VCI"

]

corr_matrix = df[corr_features].corr(method="spearman")

# =====================================================================
# STEP 10: SPEARMAN P-VALUE MATRIX
# =====================================================================
print("\nStep 10: Computing Spearman significance matrix...")

p_matrix = pd.DataFrame(
    np.ones((len(corr_features), len(corr_features))),
    index=corr_features,
    columns=corr_features
)

for i in range(len(corr_features)):
    for j in range(len(corr_features)):

        _, p = spearmanr(
            df[corr_features[i]],
            df[corr_features[j]]
        )

        p_matrix.iloc[i, j] = p

print("P-value matrix computed.")

# =====================================================================
# STEP 11: SAVE CORRELATION MATRICES
# =====================================================================
print("\nStep 11: Exporting correlation matrices...")

corr_csv = os.path.join(
    ml_dir,
    "spearman_correlation_matrix.csv"
)

pvalue_csv = os.path.join(
    ml_dir,
    "spearman_pvalue_matrix.csv"
)

corr_matrix.to_csv(corr_csv)

p_matrix.to_csv(pvalue_csv)

print("Correlation CSV saved.")
print("P-value CSV saved.")

# =====================================================================
# STEP 12: PUBLICATION QUALITY HEATMAP
# =====================================================================
print("\nStep 12: Creating publication-quality heatmap...")

mask = np.triu(
    np.ones_like(corr_matrix, dtype=bool)
)

fig, ax = plt.subplots(figsize=(14,12))

cmap = sns.diverging_palette(
    230,
    20,
    as_cmap=True
)

sns.heatmap(

    corr_matrix,

    mask=mask,

    cmap=cmap,

    vmin=-0.8,

    vmax=0.8,

    center=0,

    square=True,

    linewidths=0.5,

    annot=True,

    fmt=".2f",

    annot_kws={
        "size":9,
        "weight":"bold"
    },

    cbar_kws={
        "shrink":0.75,
        "label":"Spearman Rank Correlation ($\\rho$)"
    },

    ax=ax

)

ax.set_title(
    "Spearman Rank Correlation Matrix with Lagged Drivers (Ningxia)",
    fontsize=15,
    weight="bold",
    pad=20
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
    fontsize=9
)

ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0,
    fontsize=9
)

plt.tight_layout()

heatmap_path = os.path.join(
    ml_dir,
    "spearman_lagged_heatmap.png"
)

plt.savefig(
    heatmap_path,
    dpi=600,
    bbox_inches="tight"
)

plt.close()

print("Heatmap saved successfully.")

# =====================================================================
# PART 1 COMPLETE
# =====================================================================

print("\n" + "="*70)
print("PART 1 COMPLETED SUCCESSFULLY")
print("="*70)

print(f"Current Dataset Shape : {df.shape}")

print("\nGenerated Outputs:")
print(f"1. {heatmap_path}")
print(f"2. {corr_csv}")
print(f"3. {pvalue_csv}")

print("\nThe cleaned dataframe 'df' will be used directly in Part 2.")
print("="*70)

ADVANCED PREPROCESSING PIPELINE - PART 1

Step 1: Importing libraries...

Step 2: Reading dataset...
Dataset Loaded Successfully
Rows : 7,323,209
Columns : 11

Step 3: Sorting pixel time-series...

Step 4: Generating lagged variables...
Lag variables generated successfully.

Step 5: Removing lag edge NaNs...
Removed : 104,618 rows
Remaining : 7,218,591

Step 6: Removing duplicated observations...
Removed : 0 duplicate rows

Step 7: Removing outliers using IQR...
Rows removed : 496,029
Rows remaining : 6,722,562

Step 8: Dataset sanity check...
Total Missing Values : 0
Duplicate Rows       : 0

Step 9: Computing Spearman correlation matrix...

Step 10: Computing Spearman significance matrix...
P-value matrix computed.

Step 11: Exporting correlation matrices...
Correlation CSV saved.
P-value CSV saved.

Step 12: Creating publication-quality heatmap...
Heatmap saved successfully.

PART 1 COMPLETED SUCCESSFULLY
Current Dataset Shape : (6722562, 21)

Generated Outputs:
1. E:\2_Contribution

In [18]:
# =====================================================================
# PART 2: FEATURE SCALING, CYCLIC ENCODING & FINAL EXPORT
# =====================================================================

print("\n" + "="*70)
print("ADVANCED PREPROCESSING PIPELINE - PART 2")
print("="*70)

from sklearn.preprocessing import MinMaxScaler

# =====================================================================
# STEP 13: FEATURE NORMALIZATION
# =====================================================================
print("\nStep 13: Applying Min-Max Normalization...")

# Keep these columns unchanged
protected_columns = [
    "Row_Idx",
    "Col_Idx",
    "Year",
    "Month",
    "VCI"
]

# Features to normalize
features_to_scale = [
    col for col in df.columns
    if col not in protected_columns
]

scaler = MinMaxScaler(feature_range=(0,1))

df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print(f"Successfully normalized {len(features_to_scale)} predictor variables.")

# =====================================================================
# STEP 14: CYCLIC MONTH ENCODING
# =====================================================================
print("\nStep 14: Encoding Month using cyclic transformation...")

df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)

df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)

df.drop(columns=["Month"], inplace=True)

print("Month encoded successfully.")

# =====================================================================
# STEP 15: FINAL DATASET VALIDATION
# =====================================================================
print("\nStep 15: Performing final dataset validation...")

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

print(f"Missing Values : {df.isnull().sum().sum()}")

print(f"Duplicate Rows : {df.duplicated().sum()}")

# =====================================================================
# STEP 16: VARIABLE SUMMARY
# =====================================================================
print("\nStep 16: Creating statistical summary...")

summary = pd.DataFrame({

    "Minimum": df.min(numeric_only=True),

    "Maximum": df.max(numeric_only=True),

    "Mean": df.mean(numeric_only=True),

    "Std": df.std(numeric_only=True)

})

summary_path = os.path.join(
    ml_dir,
    "feature_summary_statistics.csv"
)

summary.to_csv(summary_path)

print("Feature summary exported.")

# =====================================================================
# STEP 17: SAVE SCALER (OPTIONAL)
# =====================================================================
print("\nStep 17: Saving scaler...")

import joblib

scaler_path = os.path.join(
    ml_dir,
    "minmax_scaler.pkl"
)

joblib.dump(
    scaler,
    scaler_path
)

print("Scaler saved.")

# =====================================================================
# STEP 18: EXPORT FINAL DATASET
# =====================================================================
print("\nStep 18: Exporting final ML-ready dataset...")

output_csv = os.path.join(
    ml_dir,
    "fully_preprocessed_lagged_ml_ready.csv"
)

df.to_csv(
    output_csv,
    index=False
)

print("Dataset exported successfully.")

# =====================================================================
# STEP 19: EXPORT FEATURE LIST
# =====================================================================
print("\nStep 19: Exporting feature names...")

feature_list = pd.DataFrame({
    "Feature_Name": df.columns
})

feature_list_path = os.path.join(
    ml_dir,
    "feature_list.csv"
)

feature_list.to_csv(
    feature_list_path,
    index=False
)

print("Feature list exported.")

# =====================================================================
# STEP 20: FINAL REPORT
# =====================================================================
print("\n" + "="*70)
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("="*70)

print("\nFinal Dataset Information")
print("-------------------------")
print(f"Rows                : {df.shape[0]:,}")
print(f"Columns             : {df.shape[1]}")
print(f"Predictor Features  : {len(df.columns)-5}")
print(f"Target Variable     : VCI")

print("\nExported Files")
print("-------------------------")
print(f"1. ML Dataset")
print(f"   {output_csv}")

print(f"\n2. Feature Summary")
print(f"   {summary_path}")

print(f"\n3. Feature List")
print(f"   {feature_list_path}")

print(f"\n4. MinMax Scaler")
print(f"   {scaler_path}")

print("\nPipeline Summary")
print("-------------------------")
print("✓ Temporal lag generation")
print("✓ Missing value removal")
print("✓ Duplicate removal")
print("✓ IQR outlier filtering")
print("✓ Spearman correlation analysis")
print("✓ Correlation matrix export")
print("✓ P-value matrix export")
print("✓ Min-Max normalization")
print("✓ Cyclic month encoding")
print("✓ Feature summary generation")
print("✓ ML-ready dataset export")

print("\n" + "="*70)
print("READY FOR MACHINE LEARNING MODEL DEVELOPMENT")
print("="*70)


ADVANCED PREPROCESSING PIPELINE - PART 2

Step 13: Applying Min-Max Normalization...
Successfully normalized 16 predictor variables.

Step 14: Encoding Month using cyclic transformation...
Month encoded successfully.

Step 15: Performing final dataset validation...
Rows    : 6,722,562
Columns : 22
Missing Values : 0
Duplicate Rows : 0

Step 16: Creating statistical summary...
Feature summary exported.

Step 17: Saving scaler...
Scaler saved.

Step 18: Exporting final ML-ready dataset...
Dataset exported successfully.

Step 19: Exporting feature names...
Feature list exported.

PREPROCESSING COMPLETED SUCCESSFULLY

Final Dataset Information
-------------------------
Rows                : 6,722,562
Columns             : 22
Predictor Features  : 17
Target Variable     : VCI

Exported Files
-------------------------
1. ML Dataset
   E:\2_Contribution\Final\Preprocess\1_ML\fully_preprocessed_lagged_ml_ready.csv

2. Feature Summary
   E:\2_Contribution\Final\Preprocess\1_ML\feature_summary_